[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jseborga/recursos-ia/blob/main/notebooks/08_animaciones_manim.ipynb)

# Animaciones de física con Manim

[Manim](https://github.com/ManimCommunity/manim) es una librería de Python, de software libre (licencia MIT), para hacer animaciones matemáticas. A diferencia de un video generado por IA, aquí **cada cuadro sale de una ecuación**: primero calculamos la física con NumPy, la verificamos y recién después la animamos.

Una IA puede escribir el código de la escena (pídele «una escena de Manim que…»), pero la física la calcula y la verifica Python.

**Cómo usarlo**
1. Ejecuta la celda 1 (instalación, 2 a 4 minutos).
2. *Entorno de ejecución → Reiniciar sesión*. Es obligatorio la primera vez.
3. Ejecuta desde la celda 2 en adelante. Cada animación aparece como video debajo de su celda.

Para que la instalación sea rápida, este cuaderno **no usa LaTeX**: los textos van con `Text`. Si quieres fórmulas con `MathTex`, hay que instalar también los paquetes `texlive` (bastante más pesados).

In [ ]:
#@title 1 · Instalar Manim (luego: Entorno de ejecución → Reiniciar sesión)
!sudo apt-get -qq update
!sudo apt-get -qq install -y libcairo2-dev libpango1.0-dev ffmpeg > /dev/null
!pip -q install manim
print("Listo. Ahora: Entorno de ejecución → Reiniciar sesión, y continúa desde la celda 2.")

In [ ]:
#@title 2 · Importar y definir el control
from manim import *
import numpy as np
config.media_width = "85%"

def verificar(condicion, mensaje):
    if not condicion:
        raise AssertionError("✘ " + mensaje)
    print("✔", mensaje)
print("Manim", __import__("manim").__version__)

## 1 · Tiro parabólico en La Paz: con y sin aire

Pelota de tenis lanzada a 20 m/s y 45°. Supuestos (cámbialos si tienes mejores datos): masa 57 g, radio 3,35 cm, coeficiente de arrastre 0,55, densidad del aire en La Paz 0,85 kg/m³ y g = 9,78 m/s².

**Control antes de animar:** sin aire, el alcance de la integración RK4 debe coincidir con la fórmula v₀² sen 2θ / g.

In [ ]:
def trayectoria(v0, ang_grados, rho, g=9.78, m=0.057, r=0.0335, Cd=0.55, dt=1e-3):
    """RK4 para el tiro con arrastre cuadrático; devuelve los puntos (x, y) hasta tocar el suelo."""
    k = 0.5 * rho * Cd * np.pi * r**2 / m
    def f(s):
        x, y, vx, vy = s; v = np.hypot(vx, vy)
        return np.array([vx, vy, -k*v*vx, -g - k*v*vy])
    a = np.radians(ang_grados)
    s = np.array([0.0, 0.0, v0*np.cos(a), v0*np.sin(a)]); pts = [s[:2].copy()]
    while True:
        k1 = f(s); k2 = f(s + dt/2*k1); k3 = f(s + dt/2*k2); k4 = f(s + dt*k3)
        nuevo = s + dt/6*(k1 + 2*k2 + 2*k3 + k4)
        if nuevo[1] < 0:                       # interpolación lineal hasta y = 0
            frac = s[1] / (s[1] - nuevo[1]); pts.append(s[:2] + frac*(nuevo[:2] - s[:2])); break
        s = nuevo; pts.append(s[:2].copy())
    return np.array(pts)

sin_aire = trayectoria(20, 45, rho=0.0)
con_aire = trayectoria(20, 45, rho=0.85)
formula = 20**2 * np.sin(np.radians(90)) / 9.78
print(f"Alcance sin aire: {sin_aire[-1, 0]:.3f} m (fórmula {formula:.3f} m) · con aire: {con_aire[-1, 0]:.2f} m")
verificar(abs(sin_aire[-1, 0] - formula) / formula < 1e-4, "sin aire, RK4 reproduce v₀² sen 2θ / g")
verificar(con_aire[-1, 0] < sin_aire[-1, 0], "con aire el alcance es menor, como debe ser")

In [ ]:
%%manim -qm -v WARNING TiroParabolico
class TiroParabolico(Scene):
    def construct(self):
        ax = Axes(x_range=[0, 45, 5], y_range=[0, 12, 2], x_length=11, y_length=4.8, tips=False).to_edge(DOWN, buff=0.6)
        titulo = Text("Tiro parabólico en La Paz: sin aire y con aire", font_size=32).to_edge(UP)
        camino = lambda tr: VMobject().set_points_as_corners([ax.c2p(x, y) for x, y in tr[::20]] + [ax.c2p(*tr[-1])])
        c0 = camino(sin_aire).set_color(BLUE); c1 = camino(con_aire).set_color(ORANGE)
        p0 = Dot(ax.c2p(0, 0), color=BLUE); p1 = Dot(ax.c2p(0, 0), color=ORANGE)
        self.play(Write(titulo), Create(ax))
        self.play(Create(c0), Create(c1), MoveAlongPath(p0, c0), MoveAlongPath(p1, c1), run_time=4, rate_func=linear)
        e0 = Text(f"sin aire: {sin_aire[-1, 0]:.1f} m", font_size=26, color=BLUE).next_to(ax.c2p(sin_aire[-1, 0], 0), UP)
        e1 = Text(f"con aire: {con_aire[-1, 0]:.1f} m", font_size=26, color=ORANGE).next_to(ax.c2p(con_aire[-1, 0], 0), UP + LEFT)
        nota = Text("control: sin aire, RK4 = v0² sen 2θ / g", font_size=22, color=GREEN).to_corner(UR).shift(DOWN * 0.8)
        self.play(FadeIn(e0), FadeIn(e1), FadeIn(nota))
        self.wait(2)

## 2 · Modos normales de tres masas

Tres masas iguales unidas por resortes iguales entre dos paredes. Los modos y las frecuencias salen de los autovalores de la matriz de rigidez (como en el cuaderno 05 y en la página «Matrices que se ven»).

**Control antes de animar:** las frecuencias numéricas deben coincidir con la fórmula cerrada ωₙ = 2 sen(nπ/8), con k = m = 1.

In [ ]:
N = 3
K = 2*np.eye(N) - np.eye(N, k=1) - np.eye(N, k=-1)
w2, V = np.linalg.eigh(K)
w = np.sqrt(w2)
exacta = 2*np.sin(np.arange(1, N + 1)*np.pi/(2*(N + 1)))
print("ω numéricas:", np.round(w, 6), " · fórmula:", np.round(exacta, 6))
verificar(np.max(np.abs(w - exacta)) < 1e-12, "las frecuencias coinciden con 2 sen(nπ/8)")
verificar(np.allclose(V.T @ V, np.eye(N)), "los modos son ortonormales")

In [ ]:
%%manim -qm -v WARNING ModosNormales
class ModosNormales(Scene):
    def construct(self):
        izq, der, y0 = -5.0, 5.0, -0.5
        eq = [izq + (der - izq)*(j + 1)/(N + 1) for j in range(N)]
        t = ValueTracker(0)
        modo = {"n": 0}
        def posiciones():
            n = modo["n"]
            return [eq[j] + 0.9*V[j, n]/np.max(np.abs(V[:, n]))*np.cos(2*w[n]*t.get_value()) for j in range(N)]
        def resortes():
            xs = [izq] + posiciones() + [der]
            grupo = VGroup()
            for a, b in zip(xs[:-1], xs[1:]):
                a2, b2 = a + (0.3 if a != izq else 0), b - (0.3 if b != der else 0)
                zz = [np.array([a2 + (b2 - a2)*i/16, y0 + (0.15 if i % 2 else -0.15)*(0 < i < 16), 0]) for i in range(17)]
                grupo.add(VMobject(color=GREY_B).set_points_as_corners(zz))
            return grupo
        masas = always_redraw(lambda: VGroup(*[Square(0.6, fill_opacity=1, color=ORANGE).move_to([x, y0, 0]) for x in posiciones()]))
        muelles = always_redraw(resortes)
        paredes = VGroup(Rectangle(width=0.15, height=1.6, fill_opacity=1, color=WHITE).move_to([izq - 0.08, y0, 0]),
                         Rectangle(width=0.15, height=1.6, fill_opacity=1, color=WHITE).move_to([der + 0.08, y0, 0]))
        titulo = Text("Modos normales de tres masas", font_size=34).to_edge(UP)
        self.add(paredes, muelles, masas); self.play(Write(titulo))
        for n in range(N):
            modo["n"] = n; t.set_value(0)
            etiqueta = Text(f"modo {n + 1}:  ω = {w[n]:.3f}  (fórmula {exacta[n]:.3f})", font_size=28, color=BLUE).next_to(titulo, DOWN)
            self.play(FadeIn(etiqueta), run_time=0.4)
            self.play(t.animate.set_value(2*np.pi/w[n]*1.0), run_time=3, rate_func=linear)
            self.play(FadeOut(etiqueta), run_time=0.3)
        self.wait(0.5)

## Para pedirle una escena a una IA

> «Escribe una escena de Manim Community (sin MathTex, solo `Text`) que anime los datos del arreglo `trayectoria` (columnas x, y en metros) sobre unos ejes, con un punto que recorra la curva y una etiqueta con el alcance. No calcules la física dentro de la escena: usa el arreglo.»

La última frase es la clave: **la IA dibuja, Python calcula y verifica**. Así una animación bonita no puede esconder una física equivocada.

Referencias: [Manim Community](https://github.com/ManimCommunity/manim) (licencia MIT) · instalación en Colab según el [manual de animaciones de la UNICAMP](https://explora.ft.unicamp.br/animacoes/Manual_Manim_Github.html).